# 并行实验：AFT 删失回归 (AFT Censored Regression)

**评估指标说明：**
根据论文公式 (31)，本实验的核心评估指标是 **系数的 RMSE（即 L2 范数距离）**：
$$ \text{RMSE} = \frac{1}{S} \sum_{s=1}^S \sqrt{\sum_{l=1}^p (\hat{\theta}_{l,s} - \theta_l^*)^2} $$
我们在代码中严格遵循了这一指标。

In [ ]:
import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from multiprocessing import Pool
from models.aft import generate_aft_data
from algorithms.admm import run_u_admm
from utils.eval_utils import calculate_metrics

def run_single_aft(seed, params):
    np.random.seed(seed)
    d_aft = generate_aft_data(
        m=params['m'], n=params['n'], p=params['p'], 
        pc=params['pc'], noise_type=params['noise_type'], rng_seed=seed
    )
    
    theta_true = d_aft['theta_true']
    
    # 1. Proposed (U-ADMM)
    t0 = time.time()
    theta_u_a, theta_n_a, hist_a = run_u_admm(
        d_aft, T=params['T'], W_inner=params['W_inner'], 
        rho=params['rho'], lam_t=params['lam_t'], verbose=False
    )
    time_uadmm = time.time() - t0
    theta_uadmm = theta_u_a[0]
    
    # 2. Avg MR (Naive)
    theta_avg = theta_n_a
    
    # 初始化默认值
    rmse_local, rmse_global, rmse_dgd = 0.0, 0.0, 0.0
    time_global, time_dgd = 0.0, 0.0
    
    if params.get('run_baselines', True):
        # 3. Local MR
        from algorithms.admm import init_all_nodes
        theta0_list = init_all_nodes(d_aft)
        local_rmses = [calculate_metrics(theta_true, th)[0] for th in theta0_list]
        rmse_local = np.mean(local_rmses)
        
        # 4. Pooled MR (Global)
        from algorithms.baselines import run_global_u_erm, run_dgd
        t0 = time.time()
        theta_global = run_global_u_erm(d_aft)
        time_global = time.time() - t0
        rmse_global, _ = calculate_metrics(theta_true, theta_global)
        
        # 5. D-subGD (Decentralized GD)
        t0 = time.time()
        theta_dgd = run_dgd(d_aft, T=params['T'] * params['W_inner'], lr=0.1)
        time_dgd = time.time() - t0
        rmse_dgd, _ = calculate_metrics(theta_true, theta_dgd)
    
    # Calculate metrics for Proposed and Avg
    rmse_uadmm, mae_uadmm = calculate_metrics(theta_true, theta_uadmm)
    rmse_avg, mae_avg = calculate_metrics(theta_true, theta_avg)
    
    result = {
        'seed': seed,
        'noise_type': params['noise_type'],
        'Local_RMSE': float(rmse_local),
        'Avg_RMSE': float(rmse_avg),
        'Proposed_RMSE': float(rmse_uadmm),
        'MAE': float(mae_uadmm),
        'time_Proposed': time_uadmm,
        'hist_rmse': hist_a['rmse'],
        'theta_hat': theta_uadmm.flatten().tolist(),
        'theta_true': theta_true.flatten().tolist()
    }
    if params.get('run_baselines', True):
        result['Pooled_RMSE'] = float(rmse_global)
        result['DGD_RMSE'] = float(rmse_dgd)
        result['time_Pooled'] = time_global
        result['time_DGD'] = time_dgd
    return result

In [ ]:
# ==========================================
# 可调参数区 (Hyperparameters)
# ==========================================
NUM_RUNS = 200       # 重复实验次数
NUM_WORKERS = 10     # 并行进程数

params = {
    'm': 10,         # 节点数量
    'n': 100,        # 每个节点的样本量
    'p': 5,          # 特征维度 (根据要求设为 5)
    'pc': 0.3,       # 网络连接概率
    'T': 20,         # 外层 ADMM 迭代次数 (控制全局收敛，可调 10-50)
    'W_inner': 5,    # 内层 ADMM 迭代次数 (控制本地共识，可调 1-10)
    'rho': 2.0,      # ADMM 惩罚参数 (核心参数！AFT模型可能需要较大的 rho)
    'lam_t': 0.0,    # L1 正则化系数 (如果真实参数不稀疏，设为 0)
    'noise_type': 'normal', # 噪声类型: 'normal', 'exp', 'cauchy', 't1', 't3', 'gumbel'
    'run_baselines': False # 设为 False 可以跳过耗时的 Pooled MR 和 D-subGD，方便快速调参
}

os.makedirs('exp2', exist_ok=True)
filename = f"exp2/exp2_aft_{NUM_RUNS}_{params['noise_type']}_p{params['p']}_pc_{str(params['pc']).replace('.', '')}_rho_{str(params['rho']).replace('.', '')}.json"

print(f"Starting Parallel AFT Experiments: {NUM_RUNS} runs...")
results = []
def update_progress(result):
    results.append(result)
    print(f"\rProgress: {len(results)}/{NUM_RUNS} runs completed.", end="")

with Pool(NUM_WORKERS) as pool:
    for i in range(NUM_RUNS):
        pool.apply_async(run_single_aft, args=(i, params), callback=update_progress)
    pool.close()
    pool.join()
print() # 换行

# 保存结果
with open(filename, 'w', encoding='utf-8') as f:
    json.dump({'parameters': params, 'results': results}, f, ensure_ascii=False, indent=4)
print(f"Results saved to {filename}")

In [ ]:
# ==========================================
# 实验完成
# ==========================================
print(f"\n实验完成！数据已保存至 {filename}")
print("请打开 plot_aft.ipynb 读取该文件以生成 Markdown 表格和收敛曲线。\n")
